# 4. Evaluation (Metodologia CRISP-ML)

## 4.1. Avaliação Comparativa dos Modelos Persistidos
Neste notebook, carregamos os dados do conjunto de teste e todos os modelos otimizados gravados no Google Drive na fase anterior para realizar uma avaliação multicritério detalhada e gerar o manifesto do modelo campeão (`models_manifest.json`).


In [ ]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import tensorflow as tf
from google.colab import drive
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

drive.mount('/content/drive', force_remount=True)

CAMINHO_DATA = '/content/drive/MyDrive/Projeto_Futebol/data'
PASTA_MODELS = '/content/drive/MyDrive/Projeto_Futebol/models'

# 1. Carregar dados e artefactos
df = pd.read_csv(os.path.join(CAMINHO_DATA, "dataset_futebol_preparado.csv"))
le = joblib.load(os.path.join(PASTA_MODELS, "label_encoder.joblib"))
scaler = joblib.load(os.path.join(PASTA_MODELS, "scaler.joblib"))
features = joblib.load(os.path.join(PASTA_MODELS, "features_list.joblib"))

# 2. Reconstruir conjunto de teste (20% mais recente)
if 'Diff_Forma_L5' not in df.columns:
    df['Diff_Forma_L5'] = np.abs(df['Home_Forma_L5'] - df['Away_Forma_L5'])
    df['Diff_GM_L5'] = np.abs(df['Home_Golos_Marcados_L5'] - df['Away_Golos_Marcados_L5'])
    df['Diff_GS_L5'] = np.abs(df['Home_Golos_Sofridos_L5'] - df['Away_Golos_Sofridos_L5'])

X = df[features]
y = le.transform(df['FTR'])

corte = int(len(df) * 0.80)
X_test = X.iloc[corte:]
y_test = y[corte:]
X_test_scaled = scaler.transform(X_test)

idx_empate = list(le.classes_).index("D") if "D" in le.classes_ else 1

# 3. Avaliar Modelos Guardados
 ficheiros_modelos = {
    "SVM": ("svm_opt.joblib", True),
    "Logistic Regression": ("logistic_regression_opt.joblib", True),
    "Random Forest": ("random_forest_opt.joblib", False),
    "Decision Tree": ("decision_tree_opt.joblib", False),
    "Gradient Boosting": ("gradient_boosting_opt.joblib", False),
    "Naive Bayes": ("naive_bayes_opt.joblib", True),
    "KNN": ("knn_opt.joblib", True),
    "TensorFlow Neural Net": ("tensorflow_mlp_opt.keras", True)
}

resultados = []

for nome, (ficheiro, is_scaled) in ficheiros_modelos.items():
    caminho = os.path.join(PASTA_MODELS, ficheiro)
    if not os.path.exists(caminho):
        continue

    X_te = X_test_scaled if is_scaled else X_test

    if ficheiro.endswith(".keras"):
        model = tf.keras.models.load_model(caminho)
        y_pred_prob = model.predict(X_te, verbose=0)
        y_pred = np.argmax(y_pred_prob, axis=1)
        tipo = "Deep Learning"
    else:
        model = joblib.load(caminho)
        y_pred = model.predict(X_te)
        tipo = "Machine Learning"

    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")
    prec_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec_macro = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1_empate = f1_score(y_test, y_pred, average=None, zero_division=0)[idx_empate]

    resultados.append({
        "Modelo": nome,
        "Tipo": tipo,
        "Ficheiro": ficheiro,
        "Accuracy (%)": round(acc * 100, 2),
        "F1-Score Macro (%)": round(f1_macro * 100, 2),
        "Precision Macro (%)": round(prec_macro * 100, 2),
        "Recall Macro (%)": round(rec_macro * 100, 2),
        "F1-Score Empate 'D' (%)": round(f1_empate * 100, 2)
    })

df_eval = pd.DataFrame(resultados).sort_values(by="F1-Score Macro (%)", ascending=False).reset_index(drop=True)

print("=" * 110)
print("🏆 QUADRO COMPARATIVO FINAL DOS MODELOS")
print("=" * 110)
print(df_eval[["Modelo", "Tipo", "Accuracy (%)", "F1-Score Macro (%)", "Precision Macro (%)", "Recall Macro (%)", "F1-Score Empate 'D' (%)"]].to_string(index=False))

In [1]:
from datetime import datetime

melhor = df_eval.iloc[0]

metadados = {
    "data_avaliacao": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "melhor_modelo": {
        "nome": melhor["Modelo"],
        "ficheiro": melhor["Ficheiro"],
        "accuracy": float(melhor["Accuracy (%)"]),
        "f1_score_macro": float(melhor["F1-Score Macro (%)"])
    },
    "classes": list(le.classes_),
    "features": features,
    "resumo_modelos": df_eval.to_dict(orient="records")
}

caminho_manifesto = os.path.join(PASTA_MODELS, "models_manifest.json")
with open(caminho_manifesto, "w", encoding="utf-8") as f:
    json.dump(metadados, f, indent=4, ensure_ascii=False)

print(f"\n✅ Manifesto de metadados exportado para: {caminho_manifesto}")
print(f"🏆 Modelo Campeão Confirmado: {melhor['Modelo']} ({melhor['Ficheiro']})")

NameError: name 'df_eval' is not defined